In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os

BASE_DIR = "/content/drive/MyDrive/hpc_tf_benchmarks"
os.makedirs(BASE_DIR, exist_ok=True)

In [ ]:

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"


!pip install -q tensorflow matplotlib seaborn pandas

In [ ]:
%%writefile worker_train_tf_profiled.py

import os
import sys
import time
import random
import numpy as np
import pandas as pd


INTRA = int(sys.argv[1])
INTER = int(sys.argv[2])
BATCH = int(sys.argv[3])


os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

import tensorflow as tf
tf.config.threading.set_intra_op_parallelism_threads(INTRA)
tf.config.threading.set_inter_op_parallelism_threads(INTER)

from tensorflow import keras
from tensorflow.keras import layers


SEED = 12345
tf.random.set_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)


t_data_start = time.perf_counter()

(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()
x_train = x_train.astype("float32") / 255.0
x_test  = x_test.astype("float32") / 255.0
y_train = y_train.squeeze()
y_test  = y_test.squeeze()

data_load_time = time.perf_counter() - t_data_start


t_model_start = time.perf_counter()

def resnet_block(x, filters):
    shortcut = x
    x = layers.Conv2D(filters, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.Conv2D(filters, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Add()([x, shortcut])
    x = layers.Activation("relu")(x)
    return x

def build_model():
    inputs = keras.Input(shape=(32, 32, 3))
    x = layers.Conv2D(32, 3, padding="same", use_bias=False)(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = resnet_block(x, 32)
    x = resnet_block(x, 32)
    x = layers.GlobalAveragePooling2D()(x)
    outputs = layers.Dense(10, activation="softmax")(x)
    return keras.Model(inputs, outputs)

model = build_model()

model_build_time = time.perf_counter() - t_model_start


t_compile_start = time.perf_counter()

model.compile(
    optimizer=keras.optimizers.SGD(0.05, momentum=0.9),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

compile_time = time.perf_counter() - t_compile_start


t_train_start = time.perf_counter()

model.fit(
    x_train,
    y_train,
    batch_size=BATCH,
    epochs=5,
    verbose=0
)

train_time_total = time.perf_counter() - t_train_start


t_eval_start = time.perf_counter()

_, test_acc = model.evaluate(x_test, y_test, verbose=0)

eval_time = time.perf_counter() - t_eval_start

BASE_DIR = "/content/drive/MyDrive/hpc_tf_benchmarks"
CSV_DIR = os.path.join(BASE_DIR, "results_tf", "csv")
os.makedirs(CSV_DIR, exist_ok=True)

df = pd.DataFrame([{
    "intra_threads": INTRA,
    "inter_threads": INTER,
    "batch_size": BATCH,

    "data_load_time_sec": data_load_time,
    "model_build_time_sec": model_build_time,
    "compile_time_sec": compile_time,

    "train_time_total_sec": train_time_total,
    "eval_time_sec": eval_time,

    "test_accuracy": test_acc
}])

fname = os.path.join(
    CSV_DIR,
    f"profile_intra{INTRA}_inter{INTER}_batch{BATCH}.csv"
)

df.to_csv(fname, index=False)
print("PROFILE SAVED:", fname)

Overwriting worker_train_tf_profiled.py


In [ ]:
%%writefile run_all_tf.py

import os
import subprocess
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

INTRA_LIST = [1]
INTER_LIST =  [1,2, 3, 4]
BATCH_SIZES = [64]

WORKER = "worker_train_tf_profiled.py"

BASE_DIR = "/content/drive/MyDrive/hpc_tf_benchmarks/results_tf"
CSV_DIR = os.path.join(BASE_DIR, "csv")
HM_DIR  = os.path.join(BASE_DIR, "heatmaps")

os.makedirs(CSV_DIR, exist_ok=True)
os.makedirs(HM_DIR, exist_ok=True)


for batch in BATCH_SIZES:
    for intra in INTRA_LIST:
        for inter in INTER_LIST:
            print(f"RUNNING intra={intra}, inter={inter}, batch={batch}")
            subprocess.run([
                "python3",
                WORKER,
                str(intra),
                str(inter),
                str(batch)
            ], check=True)


files = [
    os.path.join(CSV_DIR, f)
    for f in os.listdir(CSV_DIR)
    if f.endswith(".csv")
]

df = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
df.to_csv(f"{BASE_DIR}/all_profiles.csv", index=False)


for batch in BATCH_SIZES:
    sub = df[df["batch_size"] == batch]

    train_map = sub.pivot(
        index="intra_threads",
        columns="inter_threads",
        values="train_time_total_sec"
    )

    eval_map = sub.pivot(
        index="intra_threads",
        columns="inter_threads",
        values="eval_time_sec"
    )

    plt.figure(figsize=(7, 5))
    sns.heatmap(train_map, annot=True, fmt=".2f", cmap="viridis")
    plt.title(f"TF Training Time (Batch {batch})")
    plt.savefig(f"{HM_DIR}/train_time_batch{batch}.png")
    plt.close()

    plt.figure(figsize=(7, 5))
    sns.heatmap(eval_map, annot=True, fmt=".2f", cmap="magma")
    plt.title(f"TF Evaluation Time (Batch {batch})")
    plt.savefig(f"{HM_DIR}/eval_time_batch{batch}.png")
    plt.close()

print("ALL TENSORFLOW PROFILING RUNS COMPLETED")

Overwriting run_all_tf.py


In [ ]:
!python run_all_tf.py

RUNNING intra=1, inter=1, batch=64
2026-01-30 09:12:29.869078: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1769764350.019421   48168 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1769764350.074382   48168 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1769764350.142810   48168 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769764350.142870   48168 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769764350.142879   48168 computation_pl